# What will we do?
- Here, we will take example of Fare column of Titanic dataset
- Assume that we don't have the whole column. We only have some sample sets available with us.
- We will take 10 sample sets with 30 samples in each set.
- From those sample sets, we will predict `Population Mean`

In [676]:
import pandas as pd
import numpy as np

In [677]:
df = pd.read_csv(r"00_Titanic_Dataset.csv")
df.shape

(1309, 12)

In [678]:
# Removing rows with missing values from Fare column

df.dropna(subset = 'Fare', inplace = True)

In [679]:
df['Fare'].mean()
# This is the population mean
# We will take samples and predict the population mean from samples

33.29547928134557

In [680]:
df['Fare'].sample(5)

514      7.4958
5        8.4583
845      7.5500
202      6.4958
1256    69.5500
Name: Fare, dtype: float64

# Step 1: Calculate sample statistics. Sample mean and sample std_dev

In [681]:
samples = []

for i in range(10):
    x = df['Fare'].sample(30).values

    samples.append(x)

In [ ]:
# While calculating Confidence interval, we do x_bar +- (s/root(n))
# s mean sample std_dev. Here we have 10 sample sets. So which set's std_dev should we use.
# Answer is, we calculate std_dev of all 10 sample sets and then take average.

# converting to numpy array
samples = np.array(samples)

s = samples.std(axis = 1, ddof = 1)    # This will return 10 std_devs. ddof=1 means divide by n-1

print(f"""Std_dev of each sample set:
-----------------------------------
{s}

""")

s = s.mean()

print(f"""Mean of 10 std_dev: {s}""")


'''
NumPy's .std() by default divides by n (population formula), not n-1 (sample formula).
But for t-procedure, we MUST use sample SD which divides by n-1.

ddof=1 means divide by n-1

Think of it like this — ddof stands for "Delta Degrees of Freedom". 
Setting ddof=1 tells NumPy: "hey, this is a sample, subtract 1 from the denominator!"
'''

Std_dev of each sample set:
-----------------------------------
[94.49891983 47.94405961 50.88336053 37.48255519 29.68896835 33.69494688
 27.59202355 54.22475162 46.6266595  22.77898836]


Mean of 10 std_dev: 44.541523342798904


'\n--------------------------------------------------------------\nNumPy\'s .std() by default divides by n (population formula), not n-1 (sample formula).\nBut for t-procedure, we MUST use sample SD which divides by n-1.\n\nddof=1 means divide by n-1\n\nThink of it like this — ddof stands for "Delta Degrees of Freedom". \nSetting ddof=1 tells NumPy: "hey, this is a sample, subtract 1 from the denominator!"\n'

In [683]:
# Calculating sample mean

# While calculating Confidence interval, we do x_bar +- (s/root(n))
# x_bar means sample mean. Here we have 10 sample sets. So which set's mean should we use.
# Answer is, we calculate means of all 10 sample sets and then take average.

# converting to numpy array
samples = np.array(samples)

x_bar = samples.mean(axis = 1)    # This will return 10 x_bars
print(f"""Mean of each sample set:
-----------------------------------
{x_bar}

""")

x_bar = x_bar.mean()

print(f"""Mean of 10 x_bar: {x_bar}""")

Mean of each sample set:
-----------------------------------
[41.40958667 35.09791667 42.71207667 30.14555667 33.37014    28.74555333
 24.00903    37.72431    32.19499667 19.19805667]


Mean of 10 x_bar: 32.46072233333333


In [684]:
print(f"""Sample_mean: {x_bar}
Sample_std: {s}""")

Sample_mean: 32.46072233333333
Sample_std: 44.541523342798904


# Step 2: Find t value for 95% Confidence level

- n = 30, therefore `Degree of Freedom` = n-1 = 29
- 95% confidence level means `t_0.95`

- **t_0.95 = 2.045**

In [685]:
t = 2.045

# Step 3: Calculate `Standard Error`

- `Formula:` **SE = s/root(n)**, here n = 30 (sample size)

In [686]:
se = s/np.sqrt(30)
se

8.132132360164624

# Step 4: Calculate Margin of Error \

- `Formula:` **moe = t*se**, here t = 2.045 (sample size)

In [687]:
moe = t*se
moe

16.630210676536656

# Step 5: Calculate `Confidence Interval`

`Formula:` **CI = x_bar +- MOE**

In [688]:
lower_limit = x_bar - moe
upper_limit = x_bar + moe

print(f"""Confidence Interval = {lower_limit}- {upper_limit}""")

Confidence Interval = 15.830511656796673- 49.090933009869985


# Conclusion:
Confidence Interval = 15.976902583505353- 40.818538083161314

This means, no matter how many samples we take, 95% times mean will fall in this confidence interval.
Let's check this.
- We will take 100 samples sets of 35 samples and check means of all 100 samples.
- We will verify if means of 95% sample sets (Means 95 sets as we are taking 100 sets) fall in our calculated interval

In [689]:
fare_means = []
for i in range(100):
    fare_means.append(df['Fare'].sample(30).mean())

fare_means[0:5]

[39.16500333333333,
 33.304723333333335,
 31.585966666666664,
 29.989446666666666,
 21.176943333333334]

In [690]:
fare_means = np.array(fare_means).flatten()
print(fare_means)
print()

print(fare_means.shape)    # See, we are having 100 means

[39.16500333 33.30472333 31.58596667 29.98944667 21.17694333 33.73819333
 40.28249667 31.52138667 18.11125    39.90819667 30.18806    27.35528333
 50.91236333 31.19277667 24.03889    27.04666    40.04292    29.21041667
 43.40152333 18.34111    34.92819667 51.56194667 37.89277333 24.13790667
 81.80805333 52.24208667 34.58374667 24.58       18.93388667 42.57625333
 24.75638667 21.39347333 24.09819667 42.34805333 27.74250333 30.72986333
 21.39736    21.75166333 36.90222    29.21041667 35.88444    51.87180667
 38.18111333 24.51097    23.48055333 24.94500667 22.43680667 28.61722333
 24.54319333 23.82499667 26.64860667 23.62541667 24.94014    15.99722
 55.48805667 35.64889    32.66889    29.82166667 19.73236    23.4675
 42.24291333 22.22500333 33.99124333 33.09485667 27.85193667 37.52693667
 16.82666333 24.98097333 39.01472    47.88972    44.87763667 45.09430333
 32.28389    46.64735333 37.01194667 27.89847    34.74972333 24.40847333
 26.72278333 27.70472    26.25777333 39.05639    23.264303

In [691]:
count = ((fare_means>lower_limit) & (fare_means<upper_limit)).sum()
count

92